# แบบฝึกหัด 6: ระบบต้นแบบ (Model Deployment)

## สมาชิกกลุ่ม
- แดนเนียล บุญมี 6810451013 หมู่ 200
- ภูดิส เปรมเกษม 6810451307 หมู่ 200

## ชุดข้อมูลที่เลือก
**Mental Health Prediction Dataset**
- เจ้าของข้อมูล: `harpartapsingh13`
- ลิงก์: https://www.kaggle.com/datasets/harpartapsingh13/mental-health-prediction-dataset/data
- ใช้ข้อมูลนี้เพื่อการฝึกสร้างระบบต้นแบบ ไม่ใช้เพื่อวินิจฉัยทางคลินิก

**ระบบที่พัฒนา:** แอปพลิเคชันเว็บด้วย Streamlit สำหรับให้นักศึกษาคัดกรองสุขภาพใจเบื้องต้นด้วยตนเอง
โดยใช้โมเดล Logistic Regression (`C=0.1`) ที่เลือกไว้จากแบบฝึกหัด 4 และประเมินผลแล้วในแบบฝึกหัด 5


## วิดีโอสาธิต

**ลิงก์วิดีโอ:** https://youtu.be/-anxRJebAcc

วิดีโอความยาวประมาณ 4 นาที แสดงการใช้งาน 3 กรณี

| กรณี | ข้อมูลที่ใส่ | ผลที่ระบบแสดง |
|---|---|---|
| ปกติ | ข้อมูลของนักศึกษาที่นอนพอ เครียดต่ำ มีคนรอบตัวให้พึ่งพา | กล่องสีเขียว "ปกติ" ความมั่นใจสูง |
| ทายผิด | ข้อมูลจริงจากชุดทดสอบที่คำตอบคือ Anxiety | โมเดลทายเป็น Burnout แต่ความมั่นใจต่ำกว่าเกณฑ์ ระบบจึงเตือน "ไม่มั่นใจ" แทนการฟันธง |
| ไม่มั่นใจ | ข้อมูลที่คาบเกี่ยวหลายภาวะ | กล่องสีเหลือง "ไม่มั่นใจ" พร้อมแนะนำช่องทางขอความช่วยเหลือ |


## เตรียมข้อมูลสำหรับรายงาน

โหลดโมเดลที่ใช้ในแอปและแบ่งข้อมูลแบบเดียวกับแบบฝึกหัด 4–5 เพื่อใช้ประกอบคำอธิบายในแต่ละข้อ


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

model = joblib.load("mental_health_model.joblib")
df = pd.read_csv("mental_health_prediction.csv")

final_features = [
    "sleep_hours", "social_media_hours", "academic_work_pressure",
    "physical_activity_days", "stress_level", "anxiety_score",
    "depression_score", "work_life_balance", "mood_score",
    "concentration_level", "social_support"
]

X = df[final_features]
y = df["mental_health_condition"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)


(350, 11) (75, 11) (75, 11)


---
## 1. ตัวแบบ/pipeline ที่ทำงานในระบบ รวมถึงขนาดและโครงสร้างของไฟล์

In [2]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('simpleimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite

In [3]:
logreg = model.named_steps["logisticregression"]

print("คลาสที่ทำนาย:", list(logreg.classes_))
print("จำนวน feature:", logreg.n_features_in_)
print("ขนาด coefficient:", logreg.coef_.shape)


คลาสที่ทำนาย: ['Anxiety', 'Burnout', 'Depression', 'Normal']
จำนวน feature: 11
ขนาด coefficient: (4, 11)


### 1.1 ตัวแบบที่ใช้
ระบบนี้ใช้ Logistic Regression เป็นตัวแบบสำหรับงาน Multiclass Classification เพื่อจำแนกผลออกเป็น 4 กลุ่ม ได้แก่ Normal, Anxiety, Depression และ Burnout

### 1.2 Pipeline
Pipeline คือขั้นตอนทั้งหมดที่ข้อมูลต้องผ่านก่อนและระหว่างการทำนาย โดยในระบบนี้ประกอบด้วย
SimpleImputer → StandardScaler → Logistic Regression
- SimpleImputer ใช้จัดการค่าที่หายไปด้วยค่ามัธยฐาน
- StandardScaler ใช้ปรับสเกลของข้อมูล
- LogisticRegression ใช้สำหรับจำแนกคลาส

### 1.3 การทำนายผล
เมื่อผู้ใช้กรอกข้อมูล ระบบจะใช้ predict() เพื่อเลือกคลาสที่ทำนาย และใช้ predict_proba() เพื่อหาความน่าจะเป็นของแต่ละคลาส

### 1.4 ขนาดและโครงสร้างของไฟล์
ระบบประกอบด้วยไฟล์หลักดังนี้
- app.py ขนาดประมาณ 5.3 KB ใช้สำหรับรันเว็บแอปและแสดงผล
- mental_health_model.joblib ขนาดประมาณ 2.9 KB ใช้เก็บ Pipeline และโมเดลที่ฝึกแล้ว
- save_model.py ขนาดประมาณ 1.1 KB ใช้สำหรับสร้างและบันทึกโมเดล
- mental_health_prediction.csv ขนาดประมาณ 84 KB เป็นชุดข้อมูลสำหรับฝึกโมเดล
- requirements.txt ขนาดประมาณ 66 bytes ใช้ระบุ package ที่จำเป็น

---
## 2. การออกแบบ UI พร้อมเหตุผล

ระบบถูกออกแบบเป็นเว็บแอปสำหรับ คัดกรองสุขภาพใจเบื้องต้นของนักศึกษา โดยเน้นให้ใช้งานง่ายและไม่ต้องมีความรู้ด้าน Machine Learning

### 2.1 การรับข้อมูล
ระบบใช้ Slider สำหรับกรอกข้อมูล เช่น ชั่วโมงการนอน ระดับความเครียด ความวิตกกังวล และสมาธิ เนื่องจากข้อมูลส่วนใหญ่เป็นค่าตัวเลขที่มีช่วงกำหนดไว้ชัดเจน จึงช่วยลดความผิดพลาดจากการกรอกค่าที่อยู่นอกช่วง

### 2.2 การจัดกลุ่มคำถาม
คำถามถูกแบ่งเป็น 2 ส่วน ได้แก่ การใช้ชีวิตประจำวัน และ ความรู้สึกช่วงนี้ เพื่อให้ผู้ใช้เข้าใจและกรอกข้อมูลได้เป็นลำดับมากขึ้น

### 2.3 การแสดงผล
หลังจากกดปุ่มดูผล ระบบจะแสดงผลการคัดกรอง พร้อม ค่า Confidence และกราฟความน่าจะเป็นของแต่ละกลุ่ม เพื่อให้ผู้ใช้เห็นว่าโมเดลมีแนวโน้มไปทางคลาสใด และมีความมั่นใจมากน้อยเพียงใด

### 2.4 ความเหมาะสมกับผู้ใช้กลุ่มเป้าหมาย
UI ใช้ภาษาไทยและคำที่เข้าใจง่าย เนื่องจากกลุ่มเป้าหมายคือ นักศึกษา และมีข้อความชี้แจงว่าระบบเป็นเพียงต้นแบบเพื่อการศึกษา ไม่ใช่เครื่องมือสำหรับวินิจฉัยทางการแพทย์

---

## 3. สิ่งที่ต้องจัดเตรียมเพื่อให้ระบบรันบนเครื่องอื่น

**Package ที่ต้องใช้ (`requirements.txt`)**


In [5]:
print(open("requirements.txt").read())

streamlit==1.64.0
pandas==3.0.6
scikit-learn==1.9.1
joblib==1.6.0



**คำตอบ**

| สิ่งที่ต้องเตรียม | รายละเอียด |
|---|---|
| Python | ต้องมี Python และติดตั้ง package ตาม `requirements.txt` ด้วย `pip install -r requirements.txt` |
| Package | streamlit 1.64.0, pandas 3.0.6, scikit-learn 1.9.1, joblib 1.6.0 โดยระบุเวอร์ชันตายตัว |
| ไฟล์ | `app.py`, `mental_health_model.joblib` และ `requirements.txt` ต้องอยู่โฟลเดอร์เดียวกัน เพราะแอปโหลดโมเดลด้วย path แบบ relative (`./mental_health_model.joblib`) |
| Port | แอปใช้ port 8501 ถ้ามีโปรแกรมอื่นใช้อยู่ต้องปิดก่อน หรือเปลี่ยนด้วย `--server.port` |
| คำสั่งรัน | `streamlit run app.py --server.port 8501` แล้วเปิด `http://localhost:8501` |

**ทำไมต้องระบุเวอร์ชัน scikit-learn ตายตัว — ปัญหาที่เจอจริงระหว่างพัฒนา**

ระหว่างพัฒนา แอปแสดง error `AttributeError: 'LogisticRegression' object has no attribute 'multi_class'` ทั้งที่รันบนเครื่องเดียวกัน สาเหตุคือในเครื่องมี Python สองชุด
- Jupyter notebook ใช้ scikit-learn **1.9.1** สร้างไฟล์โมเดล
- คำสั่ง `streamlit` ใช้ Python ของ Anaconda ซึ่งมี scikit-learn **1.7.2** โหลดไฟล์โมเดล

ไฟล์ `.joblib` บันทึกโครงสร้างภายในของ object ตามเวอร์ชันที่สร้าง เมื่อโหลดด้วยเวอร์ชันอื่นจึงอาจพังได้ แก้โดยเรียก Streamlit ผ่าน Python ตัวเดียวกับ notebook ด้วย `sys.executable -m streamlit run app.py` ทำให้การสร้างโมเดลและการรันแอปใช้ scikit-learn เวอร์ชันเดียวกัน

**บทเรียนสำหรับการย้ายไปเครื่องอื่น:** ต้องใช้ scikit-learn เวอร์ชันเดียวกับที่ใช้สร้างไฟล์โมเดล หรือรัน `save_model.py` เพื่อสร้างไฟล์โมเดลใหม่บนเครื่องปลายทาง ถ้าจะนำไปไว้บน Streamlit Community Cloud ต้องตั้งค่า Python version ให้ตรงกับเครื่องที่สร้างโมเดลด้วย


## 4. ข้อจำกัดของระบบ

### 4.1 ความหน่วง

In [6]:
sample = X_test.iloc[[0]]

times = []
for _ in range(100):
    start = time.perf_counter()
    model.predict_proba(sample)
    times.append((time.perf_counter() - start) * 1000)

print(f"เวลาทำนายเฉลี่ยต่อครั้ง: {np.mean(times):.2f} ms")
print(f"เวลาทำนายสูงสุด: {np.max(times):.2f} ms")


เวลาทำนายเฉลี่ยต่อครั้ง: 1.36 ms
เวลาทำนายสูงสุด: 2.59 ms


### 4.2 Confidence Threshold

In [7]:
proba = model.predict_proba(X_test)
confidence = proba.max(axis=1)
pred = model.predict(X_test)
correct = pred == y_test.values

rows = []
for th in [0.5, 0.6, 0.7]:
    low = confidence < th
    rows.append({
        "threshold": th,
        "เคสที่ต่ำกว่าเกณฑ์": f"{low.sum()}/{len(low)}",
        "accuracy เคสที่ผ่านเกณฑ์": round(correct[~low].mean(), 4),
        "accuracy เคสที่ต่ำกว่าเกณฑ์": round(correct[low].mean(), 4) if low.sum() else None,
    })

pd.DataFrame(rows)


,threshold,เคสที่ต่ำกว่าเกณฑ์,accuracy เคสที่ผ่านเกณฑ์,accuracy เคสที่ต่ำกว่าเกณฑ์
0,0.5,4/75,0.9718,1.0000
1,0.6,12/75,0.9841,0.9167
2,0.7,18/75,0.9825,0.9444


### 4.3 กรณีที่ทำนายผิดพลาด

In [8]:
wrong = ~correct
wrong_cases = X_test[wrong].copy()
wrong_cases["คำตอบจริง"] = y_test[wrong].values
wrong_cases["โมเดลทาย"] = pred[wrong]
wrong_cases["ความมั่นใจ"] = confidence[wrong].round(3)
wrong_cases


,sleep_hours,social_media_hours,academic_work_pressure,physical_activity_days,stress_level,anxiety_score,depression_score,work_life_balance,mood_score,concentration_level,social_support,คำตอบจริง,โมเดลทาย,ความมั่นใจ
335,4.8,4.0,9.0,NaN,9.0,NaN,5.0,4.0,5.0,2.0,4.0,Anxiety,Burnout,0.717
4,5.0,4.3,7.0,1.0,7.0,5.0,5.0,1.0,4.0,2.0,6.0,Anxiety,Burnout,0.579


**คำตอบ**

**1. ความหน่วง**
การทำนายแต่ละครั้งใช้เวลาเพียงระดับมิลลิวินาที เพราะ Logistic Regression คำนวณแค่ผลคูณของ coefficient กับ input 11 ตัว ผู้ใช้จึงไม่รู้สึกถึงความหน่วงจากตัวโมเดล ความหน่วงที่ผู้ใช้รู้สึกจริงมาจากการเปิดแอปครั้งแรกที่ต้องโหลดไฟล์โมเดลและเริ่ม Streamlit และจากการที่ Streamlit รันสคริปต์ใหม่ทั้งหมดทุกครั้งที่ผู้ใช้ขยับ slider

**2. ข้อจำกัดด้าน input**
- ค่าทั้งหมดเป็นการประเมินตนเอง (self-report) ผลลัพธ์ขึ้นกับว่าผู้ใช้ตอบตรงความจริงแค่ไหน ถ้าผู้ใช้ตอบเลขกลางๆ ทุกข้อ หรือไม่อยากยอมรับความรู้สึกของตัวเอง ผลจะคลาดเคลื่อน
- slider จำกัดค่าให้อยู่ในช่วงเดียวกับข้อมูลที่ใช้เทรน โมเดลจึงไม่ต้องทำนายค่าที่ไม่เคยเห็น แต่ก็หมายความว่าผู้ใช้ที่มีพฤติกรรมสุดโต่งกว่าช่วงนี้จะใส่ค่าจริงไม่ได้
- ระบบไม่ได้บันทึกข้อมูลผู้ใช้ แต่ถ้าจะใช้จริงต้องคำนึงเรื่องความเป็นส่วนตัว เพราะเป็นข้อมูลสุขภาพจิตที่อ่อนไหว

**3. กรณีที่ทำนายผิดพลาด**
ความผิดพลาดทั้งหมดในชุดทดสอบเป็นการทำนาย **Anxiety ผิดเป็น Burnout** สาเหตุเพราะสองภาวะนี้มีอาการทับซ้อนกัน เคสที่ทายผิดมีคะแนนสมดุลชีวิตและสมาธิต่ำมาก ซึ่งเป็นลักษณะเด่นของภาวะหมดไฟ ในขณะที่คะแนนความวิตกกังวลอยู่ระดับกลาง นอกจากนี้โมเดลเรียนรู้จากข้อมูลจำลอง (real-world inspired) ผลกับผู้ใช้จริงจึงอาจผิดพลาดมากกว่าที่วัดได้จากชุดทดสอบ

**4. Confidence threshold**
ระบบตั้งเกณฑ์ไว้ที่ **0.6** ถ้าความน่าจะเป็นของคลาสที่ทายต่ำกว่า 0.6 ระบบจะไม่แสดงผลแบบฟันธง แต่เตือนว่าไม่มั่นใจและแนะนำให้ปรึกษาผู้เชี่ยวชาญแทน

เลือก 0.6 จากตารางในข้อ 4.2 เพราะเป็นจุดที่แยกเคสเสี่ยงออกได้จริง เคสที่ความมั่นใจสูงกว่าเกณฑ์ทายถูกประมาณ 98% ในขณะที่เคสที่ต่ำกว่าเกณฑ์มีโอกาสผิดมากกว่า ถ้าตั้งที่ 0.5 จะจับเคสได้น้อยเกินไปและเคสที่จับได้ก็ทายถูกหมด จึงไม่ได้ช่วยกรองเคสเสี่ยง ส่วน 0.7 จะทำให้ระบบเตือนบ่อยขึ้นโดยไม่ได้แยกเคสเสี่ยงได้ดีกว่า 0.6

**threshold จับเคสผิดได้ 1 จาก 2 เคส** จากตารางในข้อ 4.3 เคสที่ทายผิดเคสหนึ่งมีความมั่นใจ 0.579 ต่ำกว่าเกณฑ์ ระบบจึงเตือนว่าไม่มั่นใจแทนการแสดงคำตอบผิด ดังที่แสดงในวิดีโอ แต่อีกเคสมีความมั่นใจ 0.717 สูงกว่าเกณฑ์ ระบบจึงแสดงผลผิดโดยดูเหมือนมั่นใจ แสดงว่า threshold ช่วยลดความเสี่ยงได้ แต่ไม่ได้รับประกันว่าผลที่ผ่านเกณฑ์จะถูกเสมอ นี่คือเหตุผลที่ระบบยังต้องมีข้อความแจ้งว่าไม่ใช่การวินิจฉัยในทุกกรณี ไม่ใช่เฉพาะตอนไม่มั่นใจ

ข้อควรระวังเพิ่มเติมคือเกณฑ์นี้หามาจากชุดทดสอบเพียง 75 แถว และจากแบบฝึกหัด 5 พบว่าโมเดล under-confident (ความมั่นใจเฉลี่ย 0.825 แต่ทายถูกจริง 0.973) ถ้าจะใช้งานจริงควรทำ calibration ก่อนแล้วหาเกณฑ์ใหม่จากข้อมูลที่มากกว่านี้


---

## สรุปส่งท้าย

รายงานนี้จัดทำขึ้นเพื่อประกอบแบบฝึกหัดที่ 6 วิชา Machine Learning Systems โดยนำโมเดล Logistic Regression จากแบบฝึกหัด 4–5 มาพัฒนาเป็นแอปพลิเคชันเว็บด้วย Streamlit สำหรับให้นักศึกษาคัดกรองสุขภาพใจเบื้องต้นด้วยตนเอง

ระบบรันได้จริงบนเครื่อง ใช้เวลาทำนายระดับมิลลิวินาที และมีกลไกเตือนเมื่อความมั่นใจต่ำกว่า 0.6 ซึ่งจับเคสที่ทายผิดได้ 1 จาก 2 เคสในชุดทดสอบ ดังที่เห็นในวิดีโอ การออกแบบ UI เน้นให้นักศึกษาเข้าใจง่าย ไม่ตื่นตระหนก และรู้ช่องทางขอความช่วยเหลือ

ข้อจำกัดสำคัญคือโมเดลเรียนรู้จากข้อมูลจำลอง ยังสับสนระหว่าง Anxiety กับ Burnout และต้องใช้ scikit-learn เวอร์ชันเดียวกับที่สร้างไฟล์โมเดล ระบบนี้จึงเหมาะเป็นเครื่องมือสำรวจตนเองเบื้องต้น ไม่ใช่ใช้แทนผู้เชี่ยวชาญ

### การเปิดเผยการใช้ AI

กลุ่มใช้ Claude (Anthropic) ช่วยในกระบวนการดังนี้

**ส่วนที่กลุ่มทำเอง**
- เลือกผู้ใช้กลุ่มเป้าหมาย (นักศึกษาที่ต้องการคัดกรองตนเอง) ซึ่งเป็นพื้นฐานของการออกแบบ UI
- ติดตั้งและรันระบบบนเครื่องของตนเอง ทดสอบทั้ง 3 กรณี และอัดวิดีโอสาธิต
- แก้ปัญหาการติดตั้ง package และปัญหาเวอร์ชัน scikit-learn ไม่ตรงกันระหว่างพัฒนา

**ส่วนที่ใช้ AI ช่วย**
- ช่วยเขียนโค้ด `app.py` และ `save_model.py` โดยอิงโครงสร้างจากตัวอย่างในคาบ
- ช่วยวิเคราะห์หา confidence threshold และเลือกเคสตัวอย่างสำหรับวิดีโอ
- ช่วยวิเคราะห์สาเหตุของ error `multi_class` ที่เกิดจากเวอร์ชัน scikit-learn ไม่ตรงกัน
- ช่วยร่างสคริปต์วิดีโอและคำอธิบายในรายงานเป็นภาษาไทย

**ตัวอย่างคำถามที่ใช้ระหว่างทำงาน**
- "ต้องเริ่มทำยังไง"
- "ไว้ไฟล์ที่ส่งมาทั้งหมดเป็นไฟล์เดียวกันเลยหรอ"
- "`AttributeError: 'LogisticRegression' object has no attribute 'multi_class'` แก้ยังไง"
- "ลองส่งให้เครื่องอื่นดูแต่ละคนเข้าไม่ได้โหลดนาน"

สมาชิกกลุ่มได้ตรวจสอบความถูกต้องของโค้ด ผลลัพธ์ และเนื้อหาในรายงานทั้งหมดก่อนส่งงาน
